### Pinecone을 이용한 벡터 DB 구축하기

#### ChromaDB의 단점
- 배포시 서버가 너무 자주 다운됨, 그러면서 메모리에 있던 크로마DB가 사라짐
- 대안: 클라우드DB 사용(Azure AI Search, Back-to-DB) => 문제: 백터DB를 직접 작성(해당 패키지를 이용해 코드를 다시 만들어야함)
-  LangChain에서는 자체적으로 관리되는 DB가 많기 때문에 별도의 코드를 작성하지 않고  Database 변경하면 됨

#### Pinecone을 활용한 DB 구축
- Pinecone은 고성능 벡터 데이터베이스로, AI 및 머신러닝 애플리케이션을 위한 효율적인 벡터 저장 및 검색 솔루션
- https://wikidocs.net/252407 참조
- 파이콘 랭체인 : https://python.langchain.com/v0.2/docs/integrations/vectorstores/pinecone/
- 무료로 5개까지 사용 가능(테스트용으로 사용)

In [ ]:
%pip install -qU pinecone
%pip install -qU langchain langchain-core langchain-community langchain-openai
%pip install -qU docx2txt pypdf
%pip install -qU langchain-text-splitters langchain-pinecone

- 문서를 읽어와 분활하기

In [1]:
import docx2txt
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path ="../../../실습 자료/소득세법_20260701.docx"

text = docx2txt.process(file_path)

# Document 생성
document = Document(
    page_content=text, 
    metadata={"source": file_path})

# print(document.page_content[:500])

# 텍스트 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents([document])


c:\Users\sb730\anaconda3\envs\deepl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 결과 확인
print(f"총 {len(chunks)}개의 청크로 분할되었습니다.")
print(f"첫 번째 청크 내용:\n{chunks[0].page_content}")

총 313개의 청크로 분할되었습니다.
첫 번째 청크 내용:
소득세법



소득세법

[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]

재정경제부(재산세제과(양도소득세)) 044-215-4312

재정경제부(소득세제과(근로소득)) 044-215-4216

재정경제부(금융세제과(이자소득, 배당소득)) 044-215-4233

재정경제부(소득세제과(사업소득, 기타소득)) 044-215-4217

재정경제부(국제조세제도과(비거주자)) 044-215-4651



제1장 총칙 <개정 2009. 12. 31.>



제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.

[본조신설 2009. 12. 31.]

[종전 제1조는 제2조로 이동 <2009. 12. 31.>]



제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.>

1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다.

2. “비거주자”란 거주자가 아닌 개인을 말한다.

3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.

4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다.

5. “사업자”란 사업소득이 있는 거주자를 말한다.

② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.

[본조신설 2009. 12. 31.]



제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다.

1. 거주자

2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인

② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수한 소득세를 납부할 의무를 진다.

1. 거

### Pinecone DB 사용
1. 회원가입 후 API Key 생성
2. index 생성
3. 코드 적용

In [3]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from dotenv import load_dotenv
import os

C:\Users\sb730\AppData\Local\Temp\ipykernel_34724\2533300891.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader


In [4]:
# 1. 환경 변수 로드 및 문서 준비(docx)

load_dotenv()

loader = Docx2txtLoader("../../../실습 자료/소득세법_20260701.docx")

In [5]:
# 2. 문서 로드 및 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    separators=['제', '\n\n', '\n', ' ', '']
)

# documents = loader.load()
# chunks = text_splitter.split_documents(documents)

chunks = loader.load_and_split(text_splitter=text_splitter)

In [6]:
print(f"총 {len(chunks)}개의 청크로 분할되었습니다.")

총 311개의 청크로 분할되었습니다.


In [11]:
# 3. 문서 임베딩 및 벡터 스토어 생성
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

database = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="my-tax-index"
)

In [12]:
# 4.Recursive 생성
recursive = database.as_retriever(
    search_type="similarity",  # 유사도 기반 검색
    search_kwargs={"k": 3}     # 검색할 유사 문서 개수
    )

In [13]:
# 5. Prompt 생성 => 객체 형태로 생성
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    '''
    [Identity]
    당신은 한국의 소득세법 전문가입니다.
    [Context] 제공된 내용만 이용해서 사용자의 질문에 친절하게 답변해 주세요.
    [Context]에 관련 내용이 없다면 "제공된 소득세법 문서에는 관련 내용이 없습니다."라고 답변해 주세요.
    마지막에는 반드시 [출처]를 명시해 주세요.

    [Context]
    {context}

    [Question]
    {query}
    '''
)

In [ ]:
# pip install -qU langchain-google-genai

In [14]:
# 6. LLM 생성
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gpt-4o", 
    model_provider='openai',    # "google-genai", "ollama"
    temperature=0
)

In [15]:
llm.invoke('안녕하세요.').content

'안녕하세요! 어떻게 도와드릴까요?'

In [16]:
# 7. 문자열 출력 형식
def format_response(docs):
    return "\n\n".join([f"출처: {doc.metadata['source']}\n내용: {doc.page_content}" for doc in docs])

In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


rag_chain = (
    {"context": recursive | format_response,  # recursive 결과를 format_response 함수에 전달
     "query": RunnablePassthrough()}          # 전달한 query를 그대로 사용
    | prompt_template
    | llm
    | StrOutputParser()
)

In [18]:
query = '종합과세 표준에 대해 설명해주세요'
result = rag_chain.invoke(query)

print(result)

종합소득과세표준은 거주자의 종합소득에 대한 과세표준을 의미합니다. 이는 다양한 소득 항목들, 즉 이자소득, 배당소득, 사업소득, 근로소득, 연금소득 및 기타소득의 합계액에서 종합소득공제를 적용한 금액으로 계산됩니다. 종합소득과세표준을 계산할 때는 특정 소득 항목들이 합산되지 않으며, 이는 법령에 명시된 조건에 따라 결정됩니다. 예를 들어, 「조세특례제한법」에 따라 과세되지 않는 소득이나 일용근로자의 근로소득 등은 종합소득과세표준에 포함되지 않습니다.

[출처: ../../../실습 자료/소득세법_20260701.docx]
